# Consulta léxica vs consulta semântica (busca por significado)

Este notebook foi feito para **demonstrar, na prática**, as diferenças entre:

- **Consulta léxica** (por palavras / termos — exemplo: *BM25*, *TF‑IDF*)
- **Consulta semântica** (por significado — exemplo: *embeddings* + similaridade)

Ao final, você vai conseguir responder perguntas como:
- Por que a busca por palavra falha quando há sinônimos, flexões, traduções ou ambiguidade?
- Por que a busca semântica ajuda nesses casos?
- Em quais cenários vale combinar as duas (busca híbrida)?


## Como funcionam notebooks Jupyter e como executar no Google Colab

Um **notebook Jupyter** é um documento interativo composto por **células**:

- **Células de texto (Markdown):** servem para explicar teoria, anotar passos, inserir fórmulas e links.
- **Células de código (Python):** executam trechos de código e mostram as saídas logo abaixo.

### Executando no Colab
1. Abra este arquivo no **Google Colab** (Arquivo → Upload do notebook).
2. Para executar uma célula: clique nela e pressione **Shift + Enter**.
3. Execute as células **em ordem**, de cima para baixo.
4. Se aparecer um aviso pedindo permissão para instalar pacotes, aceite.

> Dica: se você fechar a aba, o Colab pode “esquecer” a sessão. É normal precisar executar tudo novamente.


## Visão geral do que vamos construir

Vamos usar um pequeno conjunto de frases (nossos “documentos”) e implementar:

1. **Busca semântica**
   - Transformar cada frase em um **vetor numérico** (embedding)
   - Transformar a consulta em um embedding
   - Medir **similaridade (cosseno)** entre consulta e documentos
2. **Busca léxica (BM25)**
   - Tokenizar as frases (quebrar em palavras)
   - Calcular relevância por palavras e frequência (BM25)
3. **Comparação lado a lado** para várias consultas


## Teoria curta: consulta léxica (por palavras)

A consulta léxica recupera documentos comparando **termos** do texto:

- Se a consulta contém a palavra **"gato"**, documentos com **"gato"** tendem a subir no ranking.
- Métodos clássicos: **TF‑IDF** e **BM25** (muito usado em motores como Lucene/Elasticsearch).

✅ Pontos fortes
- Muito rápida e interpretável (“deu match porque tem a palavra X”)
- Excelente quando o usuário busca **termos exatos**, nomes, códigos, etc.

⚠️ Limitações típicas
- **Sinônimos** (“carro” vs “automóvel”)
- **Tradução** (português vs inglês)
- **Flexões** (“dorme”, “dormindo”, “dormiu”)
- **Ambiguidade**: mesma palavra com sentidos diferentes (ex.: **banco** = instituição financeira ou assento)


## Teoria curta: consulta semântica (por significado)

A consulta semântica usa **embeddings**: modelos transformam texto em vetores em que
textos com **sentido parecido** ficam **próximos** no espaço vetorial.

A lógica é:
- documento → embedding (vetor)
- consulta → embedding (vetor)
- rankear por **similaridade** (geralmente cosseno)

✅ Pontos fortes
- Lida melhor com **sinônimos**, paráfrases e contexto
- Pode funcionar **entre idiomas**, se o modelo for multilíngue

⚠️ Limitações típicas
- Requer modelos + infraestrutura de busca vetorial em grande escala
- Pode errar em termos raros/precisos (IDs, números, nomes muito específicos)
- Explicabilidade é menor (por que foi parecido?)


In [1]:
# Instala dependências (no Colab isso pode levar ~1-2 minutos na primeira vez)
!pip -q install sentence-transformers

import numpy as np
import pandas as pd

from IPython.display import display
from sentence_transformers import SentenceTransformer, util



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\Ladislau Lopes\Documents\curso\Projetos\Cursos\Busca semantica\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Nosso conjunto de “documentos”

Para facilitar o entendimento, vamos usar poucas frases. Observe que colocamos:
- frases **paráfrases** (mesmo sentido com palavras diferentes)
- frases em **inglês**
- a palavra **banco** em sentidos diferentes (instituição vs assento)


In [2]:
# Lista de frases (cada frase será um "documento")
sentences = [
    # Gato/sofá (paráfrases + inglês)
    "O gato está dormindo no sofá",
    "Um gato dorme em cima do sofá",
    "Há um gato dormindo no sofá da sala",
    "The cat is sleeping on the sofa",

    # Cachorro/parque (paráfrases + inglês)
    "O cachorro correu pelo parque",
    "Um cão estava correndo no parque",
    "O animal correu em um parque público",
    "The dog is running in the park",

    # Matemática
    "Eu gosto de estudar matemática",
    "Aprender matemática é importante",
    "Matemática é uma disciplina difícil",
    "Ensinar matemática exige paciência",

    # Banco (ambiguidade)
    "O banco fechou mais cedo hoje",                    # instituição
    "Sentei no banco da praça para descansar",          # assento
    "O banco central aumentou a taxa de juros",         # instituição

    # Outros tópicos aleatórios
    "A previsão do tempo indica chuva amanhã",
    "Comprei um novo teclado mecânico",
    "A fotossíntese ocorre nas folhas das plantas",
    "The stock market closed higher today",
]

docs = pd.DataFrame({"doc_id": range(len(sentences)), "texto": sentences})
docs

,doc_id,texto
0,0,O gato está dormindo no sofá
1,1,Um gato dorme em cima do sofá
2,2,Há um gato dormindo no sofá da sala
3,3,The cat is sleeping on the sofa
4,4,O cachorro correu pelo parque
5,5,Um cão estava correndo no parque
6,6,O animal correu em um parque público
7,7,The dog is running in the park
8,8,Eu gosto de estudar matemática
9,9,Aprender matemática é importante


## 2) Similaridade de cosseno (por que ela aparece sempre?)

Quando usamos embeddings, cada texto vira um vetor **v**.  
A **similaridade de cosseno** mede o “ângulo” entre dois vetores:

- `1.0` → muito semelhantes (mesma direção)
- `0.0` → sem relação
- `-1.0` → opostos (raro em embeddings de sentença)

Na prática, muita gente **normaliza** os vetores para norma 1.  
Aí o cosseno vira essencialmente um produto interno.


## 3) Busca semântica: embeddings + ranking por similaridade

Vamos usar um modelo **multilíngue** do Sentence-Transformers, para que frases em português e inglês possam se aproximar no espaço semântico.


In [3]:
# Carrega o modelo de embeddings
# Observação: o download do modelo pode demorar na primeira execução.
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# Calcula embeddings dos documentos
# - convert_to_tensor=True: fica como tensor (mais eficiente para similaridade)
# - normalize_embeddings=True: normaliza para norma 1 (bom para cosseno)
doc_embeddings = model.encode(
    sentences,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print("Nº de documentos:", len(sentences))
print("Dimensão do embedding:", doc_embeddings.shape[1])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2121.61it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Nº de documentos: 19
Dimensão do embedding: 768


In [4]:
# Exemplo rápido: similaridade entre duas frases muito parecidas
frase_1 = "O gato está dormindo no sofá"
frase_2 = "Um gato dorme em cima do sofá"

v1 = model.encode(frase_1, convert_to_tensor=True, normalize_embeddings=True)
v2 = model.encode(frase_2, convert_to_tensor=True, normalize_embeddings=True)

sim = util.cos_sim(v1, v2).item()
print(f"Similaridade (cosseno) entre:\n- {frase_1}\n- {frase_2}\n= {sim:.4f}")


Similaridade (cosseno) entre:
- O gato está dormindo no sofá
- Um gato dorme em cima do sofá
= 0.9856


### Função de recuperação semântica

Vamos criar uma função `semantic_retrieve(query)` que retorna os `top_k` documentos mais similares.


In [5]:
def semantic_retrieve(query: str, top_k: int = 5):
    """Retorna os top_k documentos mais similares à consulta (busca semântica)."""
    # Embedding da consulta
    query_emb = model.encode(query, convert_to_tensor=True, normalize_embeddings=True)

    # Similaridade da consulta com todos os documentos (vetor de scores)
    scores = util.cos_sim(query_emb, doc_embeddings)[0]  # shape: (N,)

    # Ordena do maior para o menor
    top_idx = torch_topk_indices(scores, k=top_k)

    results = []
    for i in top_idx:
        results.append((float(scores[i].item()), sentences[int(i)]))
    return results


def torch_topk_indices(scores_tensor, k: int):
    """Retorna índices dos k maiores valores de um tensor 1D (sem depender de torch.topk explicitamente)."""
    # Alguns ambientes já têm torch; Sentence-Transformers depende dele.
    import torch
    k = min(k, scores_tensor.shape[0])
    values, indices = torch.topk(scores_tensor, k=k)
    return indices.cpu().numpy().tolist()


### Testando buscas semânticas

Repare especialmente em:
- consultas em **inglês** recuperando documentos em **português**
- ambiguidade em **banco** quando acrescentamos contexto (“juros” vs “praça”)


In [6]:
queries = [
    "gato no sofá",
    "cat sleeping on sofa",
    "cão correndo no parque",
    "dog in the park",
    "matemática importante",
    "banco",             # ambíguo
    "banco juros",       # tende ao sentido financeiro
    "banco praça",       # tende ao sentido de assento
]

for q in queries:
    print("\nQUERY:", q)
    for score, text in semantic_retrieve(q, top_k=3):
        print(f"  {score:0.4f} | {text}")



QUERY: gato no sofá
  0.8859 | Um gato dorme em cima do sofá
  0.8859 | Há um gato dormindo no sofá da sala
  0.8621 | The cat is sleeping on the sofa

QUERY: cat sleeping on sofa
  0.9855 | The cat is sleeping on the sofa
  0.9730 | Um gato dorme em cima do sofá
  0.9568 | O gato está dormindo no sofá

QUERY: cão correndo no parque
  0.9870 | Um cão estava correndo no parque
  0.9789 | The dog is running in the park
  0.9652 | O cachorro correu pelo parque

QUERY: dog in the park
  0.8355 | Um cão estava correndo no parque
  0.8355 | The dog is running in the park
  0.7996 | O cachorro correu pelo parque

QUERY: matemática importante
  0.7800 | Aprender matemática é importante
  0.7040 | Matemática é uma disciplina difícil
  0.5467 | Ensinar matemática exige paciência

QUERY: banco
  0.4414 | O banco fechou mais cedo hoje
  0.3121 | O banco central aumentou a taxa de juros
  0.1700 | Matemática é uma disciplina difícil

QUERY: banco juros
  0.4434 | O banco central aumentou a taxa de

## 4) Preparação léxica: normalização + tokenização

Motores de busca léxicos fazem várias etapas (stopwords, stemming, lematização, etc.).
Aqui vamos usar uma versão **simplificada** para fins didáticos:

- colocar em minúsculas
- remover acentos
- quebrar em tokens alfanuméricos

Isso já é suficiente para demonstrar a ideia.


In [7]:
import re
import unicodedata
from collections import defaultdict, Counter

def normalize_text(text: str) -> str:
    """Normaliza texto: minúsculas + remove acentos."""
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text

def tokenize(text: str):
    """Tokeniza: retorna lista de tokens alfanuméricos."""
    text = normalize_text(text)
    return re.findall(r"[a-z0-9]+", text)

# Exemplo de tokenização
for s in sentences[:4]:
    print(s, "->", tokenize(s))


O gato está dormindo no sofá -> ['o', 'gato', 'esta', 'dormindo', 'no', 'sofa']
Um gato dorme em cima do sofá -> ['um', 'gato', 'dorme', 'em', 'cima', 'do', 'sofa']
Há um gato dormindo no sofá da sala -> ['ha', 'um', 'gato', 'dormindo', 'no', 'sofa', 'da', 'sala']
The cat is sleeping on the sofa -> ['the', 'cat', 'is', 'sleeping', 'on', 'the', 'sofa']


## 5) Busca léxica: BM25 (Okapi)

**BM25** é uma evolução do TF‑IDF com ajustes práticos:

- pontua termos raros (IDF) mais alto
- considera frequência do termo no documento (TF)
- normaliza pelo tamanho do documento

Em produção, você raramente implementa BM25 “na mão”, porque motores como
Lucene/Elasticsearch já fazem isso com índices invertidos.  
Mas implementar aqui ajuda a enxergar o mecanismo.


In [8]:
import math

class BM25Retriever:
    """Implementação de BM25 (Okapi)."""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b

        self.sentences = None
        self.tokenized_docs = None
        self.doc_term_freqs = None
        self.inverted_index = None
        self.doc_lens = None
        self.avgdl = None
        self.df = None
        self.idf = None
        self.N = 0

    def fit(self, sentences):
        """Indexa os documentos."""
        self.sentences = list(sentences)
        self.tokenized_docs = [tokenize(s) for s in self.sentences]
        self.doc_term_freqs = [Counter(tokens) for tokens in self.tokenized_docs]

        # Comprimento de cada documento (nº de tokens)
        self.doc_lens = [len(tokens) for tokens in self.tokenized_docs]
        self.N = len(self.sentences)
        self.avgdl = sum(self.doc_lens) / max(1, self.N)

        # df = em quantos docs cada termo aparece
        self.df = Counter()
        self.inverted_index = defaultdict(list)

        for doc_id, tf in enumerate(self.doc_term_freqs):
            for term in tf.keys():
                self.df[term] += 1
                self.inverted_index[term].append(doc_id)

        # idf (Okapi): log(1 + (N - df + 0.5)/(df + 0.5))
        self.idf = {}
        for term, df_t in self.df.items():
            self.idf[term] = math.log(1.0 + (self.N - df_t + 0.5) / (df_t + 0.5))

        return self

    def _score_doc(self, doc_id: int, query_terms):
        """Calcula score BM25 para um doc específico."""
        score = 0.0
        tf = self.doc_term_freqs[doc_id]
        dl = self.doc_lens[doc_id]
        denom_norm = self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))

        for term in query_terms:
            if term not in tf:
                continue
            tf_td = tf[term]
            idf_t = self.idf.get(term, 0.0)
            score += idf_t * (tf_td * (self.k1 + 1.0)) / (tf_td + denom_norm)

        return score

    def retrieve(self, query: str, top_k: int = 5):
        """Retorna top_k documentos por BM25."""
        query_terms = tokenize(query)
        if not query_terms:
            return []

        # candidatos = união dos docs que contêm algum termo da query
        candidates = set()
        for term in set(query_terms):
            candidates.update(self.inverted_index.get(term, []))

        if not candidates:
            return []

        scored = []
        for doc_id in candidates:
            s = self._score_doc(doc_id, query_terms)
            if s > 0:
                scored.append((s, doc_id))

        scored.sort(reverse=True, key=lambda x: x[0])
        return [(float(score), self.sentences[doc_id]) for score, doc_id in scored[:top_k]]

# Treina/ajusta o retriever
bm25 = BM25Retriever(k1=1.5, b=0.75).fit(sentences)


### Testando buscas léxicas (BM25)

Observe:
- consultas em inglês **só** retornam documentos que têm termos em inglês
- `banco` é ambíguo: documentos com a palavra “banco” aparecem juntos
- adicionar contexto (“juros”) ajuda o BM25 a priorizar o sentido desejado


In [9]:
# As mesmas consultas usadas na parte semântica (repetimos aqui para o bloco ficar independente)
queries = [
    "gato no sofá",
    "cat sleeping on sofa",
    "cão correndo no parque",
    "dog in the park",
    "matemática importante",
    "banco",             # ambíguo
    "banco juros",       # tende ao sentido financeiro
    "banco praça",       # tende ao sentido de assento
]

for q in queries:
    print("\nQUERY:", q)
    for score, text in bm25.retrieve(q, top_k=3):
        print(f"  {score:0.4f} | {text}")



QUERY: gato no sofá
  4.7814 | O gato está dormindo no sofá
  4.1655 | Há um gato dormindo no sofá da sala
  3.0471 | Um gato dorme em cima do sofá

QUERY: cat sleeping on sofa
  8.7255 | The cat is sleeping on the sofa
  1.5091 | O gato está dormindo no sofá
  1.4052 | Um gato dorme em cima do sofá

QUERY: cão correndo no parque
  8.5134 | Um cão estava correndo no parque
  1.9041 | O cachorro correu pelo parque
  1.6419 | O animal correu em um parque público

QUERY: dog in the park
  9.7054 | The dog is running in the park
  2.3851 | The cat is sleeping on the sofa
  1.7633 | The stock market closed higher today

QUERY: matemática importante
  4.8461 | Aprender matemática é importante
  1.7709 | Ensinar matemática exige paciência
  1.6295 | Eu gosto de estudar matemática

QUERY: banco
  1.7633 | O banco fechou mais cedo hoje
  1.6419 | Sentei no banco da praça para descansar
  1.5362 | O banco central aumentou a taxa de juros

QUERY: banco juros
  3.8191 | O banco central aumentou a

## 6) Comparação lado a lado (léxico vs semântico)

Agora vamos montar uma tabela para comparar os rankings dos dois métodos.

> Importante: como nosso conjunto de documentos é minúsculo, isso é apenas uma *demonstração*.
Em bases grandes, a busca semântica usa índices vetoriais (FAISS, HNSW, etc.) e a busca léxica usa índices invertidos.


In [10]:
def compare_retrievers(query: str, top_k: int = 5):
    sem = semantic_retrieve(query, top_k=top_k)
    lex = bm25.retrieve(query, top_k=top_k)

    # Padroniza tamanhos
    sem += [(None, None)] * (top_k - len(sem))
    lex += [(None, None)] * (top_k - len(lex))

    rows = []
    for i in range(top_k):
        sem_score, sem_text = sem[i]
        lex_score, lex_text = lex[i]
        rows.append({
            "rank": i + 1,
            "sem_score": sem_score,
            "sem_result": sem_text,
            "bm25_score": lex_score,
            "bm25_result": lex_text,
        })
    return pd.DataFrame(rows)

# Exemplos de comparação
for q in ["banco", "banco juros", "banco praça", "cat sleeping sofa", "fotossintese folhas"]:
    print("\n" + "="*90)
    print("QUERY:", q)
    display(compare_retrievers(q, top_k=5))



QUERY: banco


,rank,sem_score,sem_result,bm25_score,bm25_result
0,1,0.441393,O banco fechou mais cedo hoje,1.763315,O banco fechou mais cedo hoje
1,2,0.312063,O banco central aumentou a taxa de juros,1.641928,Sentei no banco da praça para descansar
2,3,0.169997,Matemática é uma disciplina difícil,1.536176,O banco central aumentou a taxa de juros
3,4,0.165831,Comprei um novo teclado mecânico,NaN,NaN
4,5,0.158237,Aprender matemática é importante,NaN,NaN



QUERY: banco juros


,rank,sem_score,sem_result,bm25_score,bm25_result
0,1,0.443372,O banco central aumentou a taxa de juros,3.819124,O banco central aumentou a taxa de juros
1,2,0.420730,O banco fechou mais cedo hoje,1.763315,O banco fechou mais cedo hoje
2,3,0.198462,Matemática é uma disciplina difícil,1.641928,Sentei no banco da praça para descansar
3,4,0.179314,Aprender matemática é importante,NaN,NaN
4,5,0.156260,Comprei um novo teclado mecânico,NaN,NaN



QUERY: banco praça


,rank,sem_score,sem_result,bm25_score,bm25_result
0,1,0.417908,O banco fechou mais cedo hoje,4.082034,Sentei no banco da praça para descansar
1,2,0.345280,Sentei no banco da praça para descansar,1.763315,O banco fechou mais cedo hoje
2,3,0.270817,O banco central aumentou a taxa de juros,1.536176,O banco central aumentou a taxa de juros
3,4,0.238004,O animal correu em um parque público,NaN,NaN
4,5,0.184371,A previsão do tempo indica chuva amanhã,NaN,NaN



QUERY: cat sleeping sofa


,rank,sem_score,sem_result,bm25_score,bm25_result
0,1,0.945960,The cat is sleeping on the sofa,6.285396,The cat is sleeping on the sofa
1,2,0.940520,Um gato dorme em cima do sofá,1.509067,O gato está dormindo no sofá
2,3,0.931825,Há um gato dormindo no sofá da sala,1.405182,Um gato dorme em cima do sofá
3,4,0.919462,O gato está dormindo no sofá,1.314679,Há um gato dormindo no sofá da sala
4,5,0.233676,Sentei no banco da praça para descansar,NaN,NaN



QUERY: fotossintese folhas


,rank,sem_score,sem_result,bm25_score,bm25_result
0,1,0.925155,A fotossíntese ocorre nas folhas das plantas,4.880214,A fotossíntese ocorre nas folhas das plantas
1,2,0.149393,Sentei no banco da praça para descansar,NaN,NaN
2,3,0.105286,The stock market closed higher today,NaN,NaN
3,4,0.087859,A previsão do tempo indica chuva amanhã,NaN,NaN
4,5,0.085509,Eu gosto de estudar matemática,NaN,NaN


## 7) Conclusões e próximos passos

**Quando a busca léxica (BM25) brilha**
- Termos exatos, nomes próprios, códigos, referências textuais
- Consultas curtas e muito objetivas
- Interpretação e depuração de relevância (por termos)

**Quando a busca semântica brilha**
- Parafraseamento, sinônimos, contexto e linguagem natural
- Consultas “em forma de pergunta”
- Multilíngue (se o modelo suportar)

**Busca híbrida**
Na prática, sistemas modernos costumam combinar as duas:
- BM25 para garantir “âncoras” por termos importantes
- embeddings para capturar significado
- re-rankers (opcional) para ordenar os melhores candidatos

### Exercícios sugeridos
1. Adicione 5 novas frases ao conjunto e teste as consultas.
2. Crie uma consulta ambígua e tente “desambiguar” com contexto.
3. Troque o modelo de embeddings por outro e compare resultados.
